# Generate missing $validation rules

Asks for the schema.org (or bioschemas) type(s) a property should accept, then
builds a JSON-Schema `anyOf` rule object in the same style already used throughout
the bioschemas profile drafts (e.g. `citation`, `alternateIdentifier` in Ginger's
file) -- a bare object reference (plus an array-of variant) for each schema.org
class or bioschemas CURIE, and a `string`/`string+format` shape for primitive
datatypes like `Text` or `URL`.

Meant as a follow-up to `schema-validation-logic-check`: when that skill flags a
property with no real JSON-Schema validation rules, run this notebook to draft
replacement rules for it.

**Run this notebook's kernel under the `bioschemas` conda environment**
(`Anaconda3\envs\bioschemas`). Only the standard library is used (`json`, `pathlib`,
`datetime`).

**A note on the prompt below:** if a human is running this directly in Jupyter, the
prompt cell will use `input()` to ask for the type(s) interactively. If Claude is
running this on the user's behalf, `input()` will hang in a non-interactive
execution -- so Claude should instead ask the user in chat which type(s) they want
*before* running this notebook, then set `expected_types` in the Parameters cell
directly (skipping the prompt cell's `input()` call).

## 1. Parameters

In [ ]:
import json
from datetime import date
from pathlib import Path

ROOT_DIR = Path(r"C:\Users\gtsueng\Anaconda3\envs\bioschemas\schema_validation_merger")
OUTPUT_DIR = ROOT_DIR / "output"

# Informational only -- not part of the output filename (that's always
# "<date>_expectedtypes.json"), but useful context to print alongside the result.
PROPERTY_NAME = ""

# Pre-fill this list to skip the interactive prompt below (e.g. Claude sets this
# after asking the user in chat). Leave as None to be prompted for input.
# Example: ["CreativeWork", "ImageObject", "URL", "Text", "bioschemastypesdrafts:Sample"]
expected_types = None

## 2. Prompt for the expected type(s)

Accepts a comma-separated list so multiple types can be given at once, matching how
properties like `citation` accept several possible shapes.

In [ ]:
if expected_types is None:
    raw = input(
        "Enter the expected type(s), comma-separated "
        "(e.g. CreativeWork, ImageObject, URL, Text, bioschemastypesdrafts:Sample): "
    )
    expected_types = [t.strip() for t in raw.split(",") if t.strip()]

if not expected_types:
    raise ValueError("No expected types given -- set expected_types above and rerun.")

label = f" for property '{PROPERTY_NAME}'" if PROPERTY_NAME else ""
print(f"Expected types{label}: {expected_types}")

## 3. Classify each type and build the `anyOf` rule

Schema.org primitive datatypes (`Text`, `URL`, `Number`, `Integer`, `Boolean`,
`Date`, `DateTime`, `Time`) get a plain `string`/`number`/... shape. Anything else --
a schema.org class like `CreativeWork`/`ImageObject`, or a bioschemas CURIE like
`bioschemastypesdrafts:Sample` -- is treated as an object reference: a bare
`{"@type": ..., "properties": {}, "required": []}`, left unexpanded the same way
`citation` and `alternateIdentifier` reference `CreativeWork`/`PropertyValue` without
spelling out their sub-properties.

Every type gets both a singular branch and an `array`-of-that-shape branch, so the
result supports single- or multi-valued data the same way the existing profile does,
regardless of what `owl:cardinality` ends up being set to.

In [ ]:
PRIMITIVE_TYPE_SHAPES = {
    "text": {"type": "string"},
    "url": {"type": "string", "format": "uri"},
    "number": {"type": "number"},
    "integer": {"type": "integer"},
    "boolean": {"type": "boolean"},
    "date": {"type": "string", "format": "date"},
    "datetime": {"type": "string", "format": "date-time"},
    "time": {"type": "string", "format": "time"},
}


def branches_for_type(type_name: str):
    """Return (single_branch, array_branch) for one expected type."""
    primitive = PRIMITIVE_TYPE_SHAPES.get(type_name.strip().lower())
    if primitive:
        single = dict(primitive)
        array = {"type": "array", "items": dict(primitive)}
        return single, array

    single = {
        "@type": type_name,
        "type": "object",
        "properties": {},
        "required": [],
    }
    array = {"type": "array", "items": dict(single)}
    return single, array


seen = set()
any_of = []
for type_name in expected_types:
    key = type_name.strip().lower()
    if key in seen:
        print(f"Skipping duplicate type: {type_name!r}")
        continue
    seen.add(key)
    single, array = branches_for_type(type_name)
    any_of.append(single)
    any_of.append(array)

validation_rule = {"anyOf": any_of}
print(json.dumps(validation_rule, indent=2))

## 4. Write the output file

Named `<date>_expectedtypes.json`. If a file for today already exists (e.g. this
notebook was already run once today for a different property), a numeric suffix is
added rather than silently overwriting the earlier run.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
today = date.today().isoformat()
base_name = f"{today}_expectedtypes"
output_path = OUTPUT_DIR / f"{base_name}.json"

suffix = 2
while output_path.exists():
    output_path = OUTPUT_DIR / f"{base_name}_{suffix}.json"
    suffix += 1

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(validation_rule, f, indent=2)
    f.write("\n")

print(f"Wrote validation rule to: {output_path}")